# 05 — Feature Engineering
Build ML-ready features from employee_attrition_processed.csv.
Each engineered feature has a stated statistical/business reason.

In [1]:

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PROC = r'../data/processed'
ea = pd.read_csv(f'{PROC}/employee_attrition_processed.csv', parse_dates=['JoiningDate','LastLeaveDate'])
print(f"Loaded: {ea.shape}")
print(ea.dtypes)


Loaded: (500, 25)
EmployeeID                        int64
Name                             object
Gender                           object
Age                               int64
Department                       object
JobRole                          object
EducationLevel                    int64
JoiningDate              datetime64[ns]
CountryCode                       int64
Country                          object
PhoneNumber                       int64
MonthlySalary                     int64
OvertimeHoursPerMonth             int64
LeavesTaken                       int64
LastLeaveDate            datetime64[ns]
LeaveDayName                     object
ProjectsHandled                   int64
TrainingHours                     int64
CustomerSatisfaction            float64
LastPromotionYear                 int64
YearsAtCompany                    int64
WorkLifeBalanceScore            float64
PerformanceRating                 int64
AttritionRisk                    object
AttritionRisk_Label   

In [2]:

# ── Drop non-predictive / leakage columns ──
# Name: free text, no signal
# PhoneNumber: identifier, no signal
# CountryCode/Country: not business-relevant for this model
# AttritionRisk (string): target variable — keep AttritionRisk_Label instead
# LeaveDayName: derived from LastLeaveDate, redundant
drop_cols = ['Name', 'PhoneNumber', 'CountryCode', 'Country', 'AttritionRisk', 'LeaveDayName']
drop_cols = [c for c in drop_cols if c in ea.columns]
df = ea.drop(columns=drop_cols).copy()
print(f"After dropping non-predictive cols: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")


After dropping non-predictive cols: (500, 19)
Remaining columns: ['EmployeeID', 'Gender', 'Age', 'Department', 'JobRole', 'EducationLevel', 'JoiningDate', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'LastLeaveDate', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'LastPromotionYear', 'YearsAtCompany', 'WorkLifeBalanceScore', 'PerformanceRating', 'AttritionRisk_Label']


In [3]:

# ── Feature 1: tenure_adjusted_salary ──
# Reason: Raw salary alone doesn't capture whether someone is paid fairly for their tenure.
# An employee with 10 years at $40k is more underpaid (and flight-risk) than a 1-year employee at the same salary.
# This feature normalizes salary by YearsAtCompany to surface compensation inequity.
df['tenure_adjusted_salary'] = df['MonthlySalary'] / (df['YearsAtCompany'] + 1)
print("Feature 1: tenure_adjusted_salary")
print(df['tenure_adjusted_salary'].describe().round(2))


Feature 1: tenure_adjusted_salary
count      500.00
mean     14698.08
std      11171.17
min       2144.12
25%       7095.67
50%      11178.59
75%      18969.16
max      59783.67
Name: tenure_adjusted_salary, dtype: float64


In [4]:

# ── Feature 2: years_since_last_promotion ──
# Reason: Employees who haven't been promoted in many years are more likely to disengage and leave.
# This is one of the top predictors in academic HR attrition literature (Mitchell et al., 2001).
# Using a fixed reference year derived from max LastPromotionYear in data to avoid data leakage.
REFERENCE_YEAR = 2024
df['years_since_last_promotion'] = REFERENCE_YEAR - df['LastPromotionYear']
df['years_since_last_promotion'] = df['years_since_last_promotion'].clip(lower=0)
print("Feature 2: years_since_last_promotion")
print(df['years_since_last_promotion'].describe().round(2))


Feature 2: years_since_last_promotion
count    500.00
mean       3.58
std        3.27
min        0.00
25%        1.00
50%        3.00
75%        5.00
max       14.00
Name: years_since_last_promotion, dtype: float64


In [5]:

# ── Feature 3: overtime_to_projects_ratio ──
# Reason: High overtime relative to few projects = inefficiency / exploitation signal.
# High overtime with many projects = high-performer under pressure.
# The ratio captures this nuance; raw overtime alone conflates the two scenarios.
df['overtime_to_projects_ratio'] = df['OvertimeHoursPerMonth'] / (df['ProjectsHandled'] + 1)
print("Feature 3: overtime_to_projects_ratio")
print(df['overtime_to_projects_ratio'].describe().round(2))


Feature 3: overtime_to_projects_ratio
count    500.00
mean       3.05
std        3.01
min        0.00
25%        1.16
50%        2.20
75%        3.88
max       19.50
Name: overtime_to_projects_ratio, dtype: float64


In [6]:

# ── Feature 4: days_since_last_leave ──
# Reason: Employees who haven't taken leave recently may be overworked or disengaged from
# work-life balance, both correlated with burnout and attrition.
reference_date = pd.Timestamp('2024-12-31')
if 'LastLeaveDate' in df.columns:
    df['days_since_last_leave'] = (reference_date - df['LastLeaveDate']).dt.days.clip(lower=0)
    df['days_since_last_leave'] = df['days_since_last_leave'].fillna(df['days_since_last_leave'].median())
    print("Feature 4: days_since_last_leave")
    print(df['days_since_last_leave'].describe().round(2))
else:
    print("LastLeaveDate not available; skipping feature 4")


Feature 4: days_since_last_leave
count    500.00
mean     178.50
std      105.71
min        3.00
25%       88.75
50%      174.50
75%      269.00
max      365.00
Name: days_since_last_leave, dtype: float64


In [7]:

# ── Feature 5: engagement_score (composite) ──
# Reason: Composite index combining WorkLifeBalance, PerformanceRating, CustomerSatisfaction
# gives a holistic well-being signal. Component variables are on different scales so we
# standardize each before summing to avoid one dominating.
from sklearn.preprocessing import StandardScaler
scaler_check = StandardScaler()
engagement_components = df[['WorkLifeBalanceScore', 'PerformanceRating', 'CustomerSatisfaction']].copy()
engagement_scaled = scaler_check.fit_transform(engagement_components)
df['engagement_score'] = engagement_scaled.mean(axis=1)
print("Feature 5: engagement_score (standardized composite)")
print(df['engagement_score'].describe().round(4))


Feature 5: engagement_score (standardized composite)
count    500.0000
mean       0.0000
std        0.5976
min       -2.2782
25%       -0.3937
50%        0.0291
75%        0.4038
max        1.5805
Name: engagement_score, dtype: float64


In [8]:

# ── Encode categorical features ──
cat_cols = ['Gender', 'Department', 'JobRole', 'EducationLevel']
for c in cat_cols:
    df[c] = df[c].astype('category')

# One-hot encode
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)
print(f"After one-hot encoding: {df_encoded.shape}")


After one-hot encoding: (500, 47)


In [9]:

# ── Drop remaining date columns (raw dates not usable by ML directly) ──
date_cols = [c for c in df_encoded.columns if 'Date' in c or df_encoded[c].dtype == 'datetime64[ns]']
df_encoded = df_encoded.drop(columns=date_cols, errors='ignore')
print(f"After dropping date cols {date_cols}: {df_encoded.shape}")


After dropping date cols ['JoiningDate', 'LastLeaveDate']: (500, 45)


In [10]:

# ── Final feature check ──
# Ensure target is present and no NaN
assert 'AttritionRisk_Label' in df_encoded.columns, "Target missing!"
missing_after = df_encoded.isnull().sum()
if missing_after.any():
    print("Filling remaining NaNs with median...")
    for c in df_encoded.columns:
        if df_encoded[c].isnull().any() and df_encoded[c].dtype in ['float64','int64']:
            df_encoded[c] = df_encoded[c].fillna(df_encoded[c].median())

print(f"Final feature matrix: {df_encoded.shape}")
print(f"Target distribution: {df_encoded['AttritionRisk_Label'].value_counts().to_dict()}")
print("\nEngineered features summary:")
new_features = ['tenure_adjusted_salary','years_since_last_promotion','overtime_to_projects_ratio',
                'days_since_last_leave','engagement_score']
for f in new_features:
    if f in df_encoded.columns:
        print(f"  {f}: mean={df_encoded[f].mean():.3f}, std={df_encoded[f].std():.3f}")


Final feature matrix: (500, 45)
Target distribution: {0: 445, 1: 55}

Engineered features summary:
  tenure_adjusted_salary: mean=14698.082, std=11171.166
  years_since_last_promotion: mean=3.582, std=3.274
  overtime_to_projects_ratio: mean=3.052, std=3.007
  days_since_last_leave: mean=178.498, std=105.714
  engagement_score: mean=0.000, std=0.598


In [11]:

# ── Save feature matrix ──
df_encoded.to_csv(f'{PROC}/feature_matrix.csv', index=False)
print(f"Saved feature_matrix.csv: {df_encoded.shape}")
print(f"Columns: {list(df_encoded.columns)}")


Saved feature_matrix.csv: (500, 45)
Columns: ['EmployeeID', 'Age', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'LastPromotionYear', 'YearsAtCompany', 'WorkLifeBalanceScore', 'PerformanceRating', 'AttritionRisk_Label', 'tenure_adjusted_salary', 'years_since_last_promotion', 'overtime_to_projects_ratio', 'days_since_last_leave', 'engagement_score', 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Department_Finance', 'Department_Hr', 'Department_It', 'Department_Marketing', 'Department_Sales', 'Department_Support', 'JobRole_Account Manager', 'JobRole_Accountant', 'JobRole_Auditor', 'JobRole_Content Lead', 'JobRole_Developer', 'JobRole_Engineer', 'JobRole_Helpdesk', 'JobRole_Hr Executive', 'JobRole_Hr Manager', 'JobRole_Sales Executive', 'JobRole_Seo Analyst', 'JobRole_Support Engineer', 'JobRole_Tester', 'EducationLevel_1', 'EducationLevel_2', 'EducationLevel_3', 'EducationLevel_4', 'EducationLevel_5']


**Feature engineering complete.** 5 new features engineered with explicit business/statistical justification. Feature matrix saved.